#### **Advanced Analytics, KPI Calculations & Leadership Views**

**Objective**: Build complex analytical SQL queries, cohort analysis, trend detection, and driver analysis

#### Connect to Database

In [1]:
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
import mysql.connector

load_dotenv()

DB_CONFIG = {
    'host': os.getenv('MYSQL_HOST', 'localhost'),
    'user': os.getenv('MYSQL_USER'),
    'password': os.getenv('MYSQL_PASSWORD'),
    'database': 'youth_employment_db'
}

conn = mysql.connector.connect(**DB_CONFIG)
print("Connected to MySQL")

Connected to MySQL


#### Run & Test Views

In [4]:
# Test Leadership Scorecard
scorecard = pd.read_sql("SELECT * FROM vw_leadership_scorecard", conn)
print("=== Leadership Executive Scorecard ===")
print(scorecard.round(3))

# Test Cohort Analysis
cohort = pd.read_sql("SELECT * FROM vw_cohort_analysis ORDER BY enrollment_quarter DESC LIMIT 10", conn)
print("\n=== Cohort Analysis Sample ===")
print(cohort.round(3))

C:\Users\tohiba\AppData\Local\Temp\ipykernel_11536\4145488578.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  scorecard = pd.read_sql("SELECT * FROM vw_leadership_scorecard", conn)


=== Leadership Executive Scorecard ===
  country_name  total_youth  overall_placement_rate  female_placement_rate  \
0     Ethiopia         5503                   0.765                  0.769   
1        Ghana         3790                   0.778                  0.774   
2       Rwanda         3717                   0.774                  0.777   
3      Nigeria         5780                   0.791                  0.790   
4        Kenya         6210                   0.782                  0.789   

   retention_rate  avg_monthly_income  income_variation  
0           0.699             284.028           115.357  
1           0.689             281.081           114.877  
2           0.693             282.538           113.358  
3           0.686             282.966           114.671  
4           0.700             284.354           112.742  


C:\Users\tohiba\AppData\Local\Temp\ipykernel_11536\4145488578.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cohort = pd.read_sql("SELECT * FROM vw_cohort_analysis ORDER BY enrollment_quarter DESC LIMIT 10", conn)



=== Cohort Analysis Sample ===
  enrollment_quarter country_name           sector_name  cohort_size  \
0            2025-Q1        Kenya            Green Jobs          142   
1            2025-Q1        Ghana  Retail & Hospitality           70   
2            2025-Q1        Ghana        Digital Skills           51   
3            2025-Q1        Ghana            Green Jobs           87   
4            2025-Q1        Kenya        Digital Skills          107   
5            2025-Q1     Ethiopia        Digital Skills          101   
6            2025-Q1     Ethiopia  Retail & Hospitality          111   
7            2025-Q1     Ethiopia            Green Jobs          109   
8            2025-Q1      Nigeria        Digital Skills          103   
9            2025-Q1      Nigeria  Retail & Hospitality          100   

   placement_rate  retention_6m_rate  avg_income  female_percentage  
0           0.802              0.640     281.730             69.718  
1           0.857              0.68

#### Driver Analysis Query (Strategic Insights)

In [5]:
driver_analysis = pd.read_sql("""
    SELECT 
        sector_name,
        country_name,
        AVG(placed) as placement_rate,
        AVG(CASE WHEN age <= 24 THEN placed ELSE NULL END) as youth_placement_rate,
        COUNT(CASE WHEN gender = 'Female' THEN 1 END) * 100.0 / COUNT(*) as pct_female,
        AVG(monthly_income_usd) as avg_income
    FROM fact_program_performance f
    JOIN dim_country c ON f.country_key = c.country_key
    JOIN dim_sector s ON f.sector_key = s.sector_key
    GROUP BY sector_name, country_name
    ORDER BY placement_rate DESC;
""", conn)

print("=== Sector & Country Driver Analysis ===")
print(driver_analysis.round(3))

C:\Users\tohiba\AppData\Local\Temp\ipykernel_11536\411504806.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  driver_analysis = pd.read_sql("""


=== Sector & Country Driver Analysis ===
             sector_name country_name  placement_rate  youth_placement_rate  \
0             Green Jobs      Nigeria           0.816                 0.799   
1         Digital Skills       Rwanda           0.797                 0.786   
2           Agribusiness      Nigeria           0.796                 0.802   
3         Digital Skills        Kenya           0.788                 0.782   
4           Agribusiness        Kenya           0.786                 0.775   
5   Retail & Hospitality        Kenya           0.784                 0.798   
6         Digital Skills      Nigeria           0.783                 0.788   
7             Green Jobs        Ghana           0.782                 0.795   
8         Digital Skills        Ghana           0.780                 0.787   
9           Agribusiness       Rwanda           0.778                 0.796   
10            Green Jobs       Rwanda           0.777                 0.787   
11         

#### Trend Analysis (Year-over-Year)

In [6]:
trend_query = """
SELECT 
    YEAR(enrollment_date) as year,
    country_name,
    AVG(placed) as placement_rate,
    AVG(still_employed_6m) as retention_rate
FROM fact_program_performance f
JOIN dim_country c ON f.country_key = c.country_key
GROUP BY year, country_name
ORDER BY year, country_name;
"""

trend = pd.read_sql(trend_query, conn)
print("=== Year-over-Year Performance Trends ===")
print(trend.round(3))

=== Year-over-Year Performance Trends ===
    year country_name  placement_rate  retention_rate
0   2022     Ethiopia           0.775           0.669
1   2022        Ghana           0.771           0.692
2   2022        Kenya           0.785           0.703
3   2022      Nigeria           0.786           0.688
4   2022       Rwanda           0.790           0.688
5   2023     Ethiopia           0.785           0.736
6   2023        Ghana           0.787           0.671
7   2023        Kenya           0.772           0.704
8   2023      Nigeria           0.797           0.678
9   2023       Rwanda           0.761           0.688
10  2024     Ethiopia           0.738           0.706
11  2024        Ghana           0.765           0.700
12  2024        Kenya           0.794           0.703
13  2024      Nigeria           0.779           0.683
14  2024       Rwanda           0.777           0.701
15  2025     Ethiopia           0.752           0.652
16  2025        Ghana           0.829   

C:\Users\tohiba\AppData\Local\Temp\ipykernel_11536\653036320.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  trend = pd.read_sql(trend_query, conn)


#### Save Key Results

In [7]:
scorecard.to_csv('../data/processed/leadership_scorecard.csv', index=False)
driver_analysis.to_csv('../data/processed/driver_analysis.csv', index=False)

print("Advanced analytics views and exports completed")
conn.close()

Advanced analytics views and exports completed
